# Trending news for your watchlist and entities of interest

**Powered by [Bigdata.com](https://bigdata.com)**

**The question:** *"Which stories about my companies and the entities I follow are trending right now?"*

"What's the news on X" is a search problem (Bigdata.com fast search answers it). This notebook answers a
different question: **what is trending**, meaning the stories that many outlets are writing about *right now*,
for entities that are unusually loud compared with their normal level of coverage.

### What "trending" means here

Bigdata.com does **not** capture reader engagement (clicks, reads, shares). Trending is measured from
**publishing momentum**, which is a strong proxy for attention:

| Signal | Question it answers | Measured from |
|---|---|---|
| **Breadth** | How many articles and *distinct outlets* are covering the **same story**? (the "engagement around similar articles") | articles grouped into stories |
| **Acceleration** | Is this entity covered more than usual right now? (`spike = now ÷ normal`) | `/v1/search/volume` |
| **Recency** | How fresh is the latest coverage? | article timestamps |

A story needs **at least 2 independent outlets** to count as trending. Stories with strong positive or negative
sentiment from credible outlets that have *not yet* built wide coverage are shown separately as **Emerging**.

> If you have your own click or read data, it can be blended in later through the
> `external_engagement_score` hook in `trending/rank.py` (default 0).

### What this notebook does

| Step | What happens | Uses an AI model? |
|---|---|---|
| 1. Resolve | Tickers and names → Bigdata.com entity IDs (Knowledge Graph) | No |
| 2. Momentum | Coverage now vs. the entity's normal baseline, number of outlets, sentiment | No |
| 3. Fetch | Pull the current window's news articles for each entity | No |
| 4. Group | Group articles that report the **same event** into one story | Yes (falls back to text similarity) |
| 5. Enrich | Story title, one-line summary, direction, driver, materiality | Yes (optional) |
| 6. Rank | Trend score; separate **Emerging** list for strong-sentiment early signals | No |
| 7. Report | Self-contained HTML report + CSV files in `outputs/` | No |

All counts, momentum figures and sentiment values are computed from Bigdata.com data. The AI model only groups,
titles and summarises articles. Run the cells top to bottom (about 2 minutes live, a few seconds in replay mode).
Each section shows its typical duration, and every code cell prints how long it took (`⏱`).

In [1]:
# Setup: make the shared `trending` package importable, tidy pandas output, time every cell.
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from trending.timing import cell_timer
cell_timer.start()   # from here on, every cell ends with "⏱ <seconds>"; the last cell shows a summary

import pandas as pd
from IPython.display import HTML, display

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

from trending import config as cfg
from trending.pipeline import start, build_stories, finish, llm_label
from trending.resolve import resolve_all, entity_table
from trending.momentum import compute_momentum, momentum_table
from trending.rank import rank_trending, rank_emerging
from trending import render

⏱ 0.0 s


## 1. Configuration

Edit the lists below to change what is tracked. Everything else (weights, thresholds, windows) lives in
`trending/config.py` and can also be overridden here, e.g. `cfg.SENTIMENT_EXTREME = 0.4`.

* **Run mode:** live when `BIGDATA_API_KEY` is set; otherwise the notebook replays the saved run in `fixtures/`
  (no keys, no network). Set `REPLAY=1` in the environment to force replay.
* **LLM:** Anthropic Claude by default (`ANTHROPIC_API_KEY`); OpenAI is also supported (`LLM_PROVIDER=openai`).
  Set `cfg.USE_LLM = False` to run without any AI model.

> ⏱ **Typical time:** live under 1 s · replay under 1 s

In [2]:
# 1. Configuration and run mode
WATCHLIST = ["AAPL", "TSLA", "NVDA", "MSFT", "AMZN", "META", "GOOGL"]   # tickers
ENTITIES_OF_INTEREST = ["OpenAI", "Federal Reserve"]                    # free text, resolved via the Knowledge Graph

cfg.WINDOW = "1D"          # "1D" = last 24h vs prior 7 days  |  "1W" = last 7 days vs prior 28 days
cfg.USE_LLM = True         # False = lexical story grouping, no AI model
ENABLE_DISCOVERY = False   # True = also show companies trending beyond your watchlist (section 8)

# Emerging list: |mean chunk sentiment| >= 0.5 (see README, "Tuning")
cfg.SENTIMENT_MODE = "mean"
cfg.SENTIMENT_EXTREME = 0.5

cfg.REPLAY = cfg.REPLAY or not cfg.BIGDATA_API_KEY
client, llm, win = start("watchlist")

Mode: LIVE · window 1D: 2026-10-12 22:40 → 2026-10-13 22:40 UTC
LLM: LLM(provider='openai', model='gpt-4.1')
⏱ 0.0 s


## 2. Resolve entities (Knowledge Graph)

Each ticker or name is mapped to a Bigdata.com entity ID. IDs only ever come from the Knowledge Graph (never
invented). When a name matches several entities, the candidates are listed and the one with the most news
coverage over the past week is chosen (→). Check this table: it is what everything below is filtered on.

> ⏱ **Typical time:** live under 1 s (a few seconds for names not seen before) · replay under 1 s

In [3]:
# 2. Resolve entities
entities = resolve_all(client, tickers=WATCHLIST, free_text=ENTITIES_OF_INTEREST, as_of=win.as_of)
entity_table(entities)

,input,label,entity_id,name,kg_type,kg_category,mode,how
0,AAPL,AAPL,D8442A,Apple Inc.,PUBLIC,companies,entity,KG ticker match (matches verified default)
1,TSLA,TSLA,DD3BB1,Tesla Inc.,PUBLIC,companies,entity,KG ticker match (matches verified default)
2,NVDA,NVDA,E09E2B,NVIDIA Corp.,PUBLIC,companies,entity,KG ticker match (matches verified default)
3,MSFT,MSFT,228D42,Microsoft Corp.,PUBLIC,companies,entity,KG ticker match (matches verified default)
4,AMZN,AMZN,0157B1,Amazon.com Inc.,PUBLIC,companies,entity,KG ticker match (matches verified default)
5,META,META,12E454,Meta Platforms Inc.,PUBLIC,companies,entity,KG ticker match (matches verified default)
6,GOOGL,GOOGL,4A6F00,Alphabet Inc.,PUBLIC,companies,entity,KG ticker match (matches verified default)
7,OpenAI,OpenAI,UWCVKI,OpenAI Foundation,PRIVATE,companies,entity,KG companies: best name match with most news
8,Federal Reserve,Federal Reserve,F38FC4,United States Federal Reserve,Central Bank,organizations,entity,KG organizations: best name match with most news


⏱ 0.0 s


## 3. Momentum: which of my names are unusually loud today?

For each entity, two calls to `/v1/search/volume` (news only):

* **current_docs**: documents in the current window; **distinct_sources**: how many outlets published them.
* **baseline_per_period**: the entity's normal volume for a window of the same length (average of the baseline days).
* **spike** = (current_docs + 1) / (baseline_per_period + 1). Above 1 means louder than usual.
* **sentiment**: Bigdata.com's average sentiment (-1 to +1); **trend**: daily document counts for the last 8 days
  (the last day is still in progress).

> ⏱ **Typical time:** live about 3 s · replay under 1 s

In [4]:
# 3. Momentum
def show_momentum(df):
    """Momentum table with a bar on the spike column (no extra dependencies)."""
    return (df.style
              .format({"spike": "{:.2f}×", "baseline_per_period": "{:,.0f}", "current_docs": "{:,}",
                       "sentiment": "{:+.2f}"}, na_rep="–")
              .bar(subset=["spike"], color="#f0a35e", vmin=0)
              .hide(axis="index"))

momentum = compute_momentum(client, entities, win)
show_momentum(momentum_table(momentum, entities))

entity,name,spike,current_docs,baseline_per_period,distinct_sources,sentiment,trend,mode
AAPL,Apple Inc.,2.94×,"1,540",523,425,+0.12,▃▂▃▃▁▁▃█,entity
META,Meta Platforms Inc.,2.22×,568,256,275,-0.02,▄▆▄▄▁▁▅█,entity
NVDA,NVIDIA Corp.,1.42×,615,433,193,+0.15,▅▇█▄▁▁▅▇,entity
OpenAI,OpenAI Foundation,1.27×,595,469,430,-0.44,▄▃▅█▁▁▂▄,entity
TSLA,Tesla Inc.,1.07×,169,157,85,-0.13,▆▇▅█▃▁▅▅,entity
GOOGL,Alphabet Inc.,1.04×,639,615,280,+0.02,▇▆█▃▁▁▄▄,entity
MSFT,Microsoft Corp.,1.00×,451,453,218,+0.02,▁▂█▃▁▁▃▂,entity
Federal Reserve,United States Federal Reserve,0.83×,410,495,117,-0.14,▁▇▄█▁▁▂▃,entity
AMZN,Amazon.com Inc.,0.81×,464,573,198,+0.07,█▅▄▂▁▁▂▃,entity


⏱ 3.0 s


## 4. Find the trending stories

Fetch the current window's articles → group articles reporting the same event into stories → enrich the leading
stories → score. Progress is printed for each step. This is the longest step: most of it is the AI model.

> ⏱ **Typical time:** live about 1-2 min · replay under 1 s

In [5]:
# 4. Fetch, group, enrich and rank stories
stories, articles, _ = build_stories(client, llm, entities, win, momentum)
trending = rank_trending(stories)

cols = ["rank", "title", "entities", "articles", "sources", "spike", "direction", "driver", "materiality",
        "avg_sentiment", "trend_score"]
trending_df = render.stories_frame(trending, "trending")
trending_df[cols]

Step 3  fetched 1785 unique articles: AAPL 216, TSLA 127, NVDA 211, MSFT 232, AMZN 216, META 220, GOOGL 219, OpenAI 139, Federal Reserve 205  (6s)


Step 4  grouped into 727 stories, 200 covered by ≥2 sources  (101s)


        merged 4 duplicate stories among the top 60


Step 5  enriched 60 stories (title, summary, direction, driver, materiality)  (107s)
Step 6  scored all stories


,rank,title,entities,articles,sources,spike,direction,driver,materiality,avg_sentiment,trend_score
0,1,OpenAI Employee Arrested for Planting Recording Device in Conference Room,OpenAI,98,63,1.27,n/a,employee arrest,3,-0.719,2.6376
1,2,Apple Launches HomeView Smart Display to Compete in Smart Home Market,AAPL,61,33,2.94,up,product launch,4,0.326,2.5615
2,3,Meta Unveils Muse Wonder AI Models to Rival OpenAI and Anthropic,META,45,34,2.22,up,AI product launch,4,0.319,2.4020
3,4,Microsoft Unveils Maia 300 AI Chip to Reduce Nvidia Reliance,MSFT,49,36,1.00,up,AI chip launch,4,0.496,2.2496
4,5,Apple Launches Upgraded Apple TV 4K and HomePod mini 2,AAPL,35,20,2.94,up,product launch,3,0.413,2.2332
5,6,Apple and LG Launch Jointly Developed Smart Home IoT Devices,AAPL,29,21,2.94,up,partnership launch,3,0.374,2.2223
6,7,Microsoft President Urges Urgent AI Safety Standards and Regulations,MSFT,29,22,1.00,n/a,regulatory advocacy,2,-0.463,1.9435
7,8,Nvidia Invests in Four French AI Startups to Expand in Europe,NVDA,22,18,1.42,up,strategic investment,3,0.376,1.9195
8,9,Nvidia's China AI Chip Market Share Drops Sharply Amid Competition,NVDA,23,15,1.42,down,China market share loss,4,-0.614,1.8955
9,10,"Google Launches Gemini 4 Argon AI Model Globally for Developers, Businesses",GOOGL,21,19,1.04,up,AI product launch,4,0.116,1.8639


⏱ 1 min 47 s


## 5. Emerging signals: strong sentiment, limited coverage so far

**Trending** answers *"what is everyone writing about?"* A strongly positive or negative story that only one or two
credible outlets have covered so far will not rank there yet, so it is captured here instead.

A story qualifies when its Bigdata.com sentiment is extreme (|sentiment| ≥ `SENTIMENT_EXTREME`), it comes from a
top-tier outlet (source rank 1-2) if it has a single source, it is inside the current window, and (with the AI model
on) its materiality is 4 or 5. It never repeats a Trending story.
`emerging score = |sentiment| × materiality/5 × recency`.

> ⏱ **Typical time:** live under 1 s · replay under 1 s

In [6]:
# 5. Emerging signals
emerging = rank_emerging(stories, trending, win, use_llm=llm is not None)

emerging_df = render.stories_frame(emerging, "emerging")
emerging_df[["rank", "title", "entities", "sentiment_signal", "sources", "best_source_rank",
             "representative_source", "driver", "materiality", "emerging_score"]]

,rank,title,entities,sentiment_signal,sources,best_source_rank,representative_source,driver,materiality,emerging_score
0,1,Alphabet Faces Securities Fraud Class Action Over AI Model Launch Delays,GOOGL,-0.668,4,1,ValueSpectrum,securities lawsuit,4,0.4955
1,2,Microsoft Suspended From US Green Card Program Amid Visa Controversy,MSFT,-0.647,10,1,New York Post (NYPost),visa program suspension,4,0.4750
2,3,LG and Apple Announce Smart Home Alliance and New IoT Devices,AAPL,0.550,1,1,Segye,product collaboration,4,0.4302
3,4,UK Chancellor Targets Amazon in £1B Online Marketplace Tax Crackdown,AMZN,-0.580,1,1,Benzinga,tax policy,4,0.4265
4,5,Apple Shares Surge Despite Wall Street Doubts on Growth Potential,AAPL,0.710,1,2,Bloomberg Línea,product launch,4,0.4198
5,6,TSMC's 2nm Process Wins Orders From AMD and Microsoft,MSFT,0.607,2,1,Chinatimes,advanced chip orders,4,0.4000
6,7,Nvidia Stock Supported by Strong AI Tailwinds and Analyst Optimism,NVDA,0.738,2,2,Yahoo! Finance,AI demand outlook,4,0.3750
7,8,OpenAI Sued Over Alleged Collection of Minors' Data Without Consent,OpenAI,-0.649,4,1,Bloomberg Law,privacy lawsuit,4,0.3654
8,9,Tesla Target Lowered Amid Spending and Sales Concerns,TSLA,-0.528,5,2,AOL.com,spending and sales decline,4,0.3649
9,10,Amazon Faces £6.3 Billion UK Price Inflation Class Action,AMZN,-0.720,1,2,Law360,antitrust class action,4,0.3595


⏱ 0.0 s


## 6. The report

The report below is also saved as a single self-contained HTML file in `outputs/` (open it in any browser or attach
it to an email), together with `*_trending.csv` and `*_emerging.csv`. Click **Show all … articles** under a story to
see every article in it, with links.

> ⏱ **Typical time:** live under 1 s · replay under 1 s

In [7]:
# 6. Build and save the report
ctx = render.ReportContext(win=win, momentum=momentum, names={e.key: e.name for e in entities})
scope = f"{len(WATCHLIST)} watchlist names + " + ", ".join(ENTITIES_OF_INTEREST)
TITLE = "Trending news: watchlist & entities of interest"

def build_report(discovery=None):
    return render.watchlist_report(title=TITLE, scope=scope, ctx=ctx, trending=trending, emerging=emerging,
                                   llm_label=llm_label(llm), discovery=discovery)

report = build_report()
paths = render.save_outputs(report, TITLE, {"trending": trending_df, "emerging": emerging_df},
                            "watchlist", day=win.as_of)
for kind, p in paths.items():
    print(f"saved {kind:<9} → {os.path.relpath(p, ROOT)}")
display(HTML(report))

saved html      → outputs/watchlist_trending_2026-10-13.html
saved trending  → outputs/watchlist_2026-10-13_trending.csv
saved emerging  → outputs/watchlist_2026-10-13_emerging.csv


⏱ 0.0 s


## 7. Also trending beyond your watchlist (optional)

Set `ENABLE_DISCOVERY = True` in the configuration cell to run this (about 1 extra minute).

1. **Shortlist:** `/v1/search/co-mentions/entities` lists the companies co-mentioned in today's news versus the
   baseline period, ranked by `(current + 1) / (baseline + 1) × log(1 + current)`. Your watchlist and generic
   entities (exchanges, news agencies, …) are excluded.
2. **Confirm:** each shortlisted company's momentum is measured with `/v1/search/volume`, exactly like your
   watchlist, and the 10 with the strongest `spike × log(1 + documents)` are kept.
3. Those 10 run through the same story steps and are added to the report as a second section.

> ⏱ **Typical time:** live about 1 min when switched on, under 1 s when off · replay under 1 s

In [8]:
# 7. Discovery (optional)
if ENABLE_DISCOVERY:
    from trending.pipeline import discover_companies

    found, d_momentum = discover_companies(client, win, exclude_ids={e.id for e in entities})
    d_stories, _, _ = build_stories(client, llm, found, win, d_momentum)
    d_trending = rank_trending(d_stories, top_n=10)
    d_ctx = render.ReportContext(win=win, momentum=d_momentum, names={e.key: e.name for e in found})
    report = build_report({"ctx": d_ctx, "trending": d_trending})
    d_df = render.stories_frame(d_trending, "trending")
    paths = render.save_outputs(report, TITLE, {"trending": trending_df, "emerging": emerging_df,
                                                "discovery": d_df}, "watchlist", day=win.as_of)
    display(HTML(report))
else:
    print("Discovery is off (set ENABLE_DISCOVERY = True in the configuration cell).")

Discovery is off (set ENABLE_DISCOVERY = True in the configuration cell).
⏱ 0.0 s


In [9]:
# Save this run's responses to fixtures/ (offline replay)
finish(client, llm)

Bigdata.com query units used this session: 362.3, LLM calls 44 (564 s summed over parallel calls, 0 from cache)
Fixtures saved under fixtures/watchlist/ (removed 58 stale files).
⏱ 0.0 s


## Run time

Every code cell above ends with `⏱`: how long it took in this run. The table adds them up per step. Live runs are
dominated by the AI steps (grouping and enrichment) and by Bigdata.com calls; replay runs take a few seconds.

In [10]:
# Run time: how long each step took in this run
cell_timer.summary()

,step,time,share,seconds
0,Setup,0.0 s,0%,0.0
1,1. Configuration and run mode,0.0 s,0%,0.0
2,2. Resolve entities,0.0 s,0%,0.0
3,3. Momentum,3.0 s,3%,3.0
4,"4. Fetch, group, enrich and rank stories",1 min 47 s,97%,106.8
5,5. Emerging signals,0.0 s,0%,0.0
6,6. Build and save the report,0.0 s,0%,0.0
7,7. Discovery (optional),0.0 s,0%,0.0
8,Save this run's responses to fixtures/ (offline replay),0.0 s,0%,0.0
9,Total,1 min 50 s,100%,109.8


## 8. How to adapt

| To… | Change |
|---|---|
| Track other companies | Edit `WATCHLIST` (tickers) or `ENTITIES_OF_INTEREST` (any name: companies, organisations, people, places, concepts) |
| Look at the past week instead of the past day | `cfg.WINDOW = "1W"` (last 7 days vs. prior 28 days) |
| Re-run a specific moment | `cfg.AS_OF = "2026-10-13T18:40:00Z"` |
| Change how stories are ranked | `cfg.WEIGHTS` (articles, sources, spike, recency, materiality) |
| Require more outlets before a story trends | `cfg.TRENDING_MIN_SOURCES` (default 2) |
| Make Emerging stricter / looser | `cfg.SENTIMENT_EXTREME`, `cfg.SENTIMENT_MODE` (`"mean"` or `"max_abs"`), `cfg.EMERGING_MAX_SOURCE_RANK`, `cfg.EMERGING_MIN_MATERIALITY` |
| Include articles that only *mention* an entity | `cfg.SEARCH_IN = "ALL"` (default `"HEADLINE"` = articles *about* the entity) |
| Retrieve more articles per entity | `cfg.FETCH_SLICES = 2` (or more), at a higher API cost |
| Blend in your own click / read data | implement `external_engagement_score(story)` in `trending/rank.py` |
| Run without an AI model | `cfg.USE_LLM = False` |

See `README.md` for the full list of settings and the limitations.